In [2]:
import pandas as pd
from rdkit import Chem
from dataclasses import dataclass
from typing import Optional

In [ ]:
AA20 = set("ACDEFGHIKLMNPQRSTVWY")
DNA  = set("ACGT")
RNA  = set("ACGU")
NUC  = set("ACGTU")

NUC_AMBIG = set("NRYSWKMBDHVX")
NUC_LIKE  = NUC | NUC_AMBIG

@dataclass(frozen=True)
class SeqCheck:
    kind: str                 # "DNA"|"RNA"|"NUCLEIC_MIXED"|"NUCLEIC_AMBIG"|"AA"|"SMILES"|"INVALID"
    length: int
    invalid_chars: str
    canonical: Optional[str]

def try_canonical_smiles(s: str) -> Optional[str]:
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

def classify_sequence(s: str, allow_aa_uo: bool = False) -> SeqCheck:
    if s is None:
        return SeqCheck("INVALID", 0, "None", None)

    s = s.strip()
    if not s:
        return SeqCheck("INVALID", 0, "EMPTY", None)

    s_smiles = s
    s_seq = s.upper()
    chars = set(s_seq)

    if chars <= NUC_LIKE:
        amb = chars & NUC_AMBIG
        if amb:
            return SeqCheck("INVALID", len(s_seq), "".join(sorted(amb)), None)

        # strictly A/C/G/T/U only
        has_t = "T" in chars
        has_u = "U" in chars
        if has_t and has_u:
            return SeqCheck("NUCLEIC_MIXED", len(s_seq), "", None)
        if has_t:
            return SeqCheck("DNA", len(s_seq), "", None)
        if has_u:
            return SeqCheck("RNA", len(s_seq), "", None)
        return SeqCheck("NUCLEIC_AMBIG", len(s_seq), "", None)  # only A/C/G

    # AA
    AA = AA20 | (set("UO") if allow_aa_uo else set())
    if chars <= AA:
        return SeqCheck("AA", len(s_seq), "", None)

    # SMILES
    can = try_canonical_smiles(s_smiles)
    if can is not None:
        return SeqCheck("SMILES", len(s_smiles), "", can)

    # INVALID
    allowed = AA | NUC  
    invalid = "".join(sorted(chars - allowed))
    return SeqCheck("INVALID", len(s_seq), invalid, None)

In [ ]:
base = " "
#miRNA
path_miRNA_target = f"{base}/data/row/miRNA-target-1.xlsx"
path_miRNA_annotation = f"{base}/data/row/miRNA-annotation-1.xlsx"
path_miRNA_RNA = f"{base}/data/row/miRNA-RNA-1.xlsx"

#Repeats
path_repeats_target = f"{base}/data/row/Repeats-target-1.xlsx"
path_repeats_annotation = f"{base}/data/row/Repeats-annotation-1.xlsx"
path_repeats_RNA = f"{base}/data/row/Repeats-RNA-1.xlsx"

#Ribosomal
path_ribosomal_target = f"{base}/data/row/Ribosomal-target-1.xlsx"
path_ribosomal_annotation = f"{base}/data/row/Ribosomal-annotation-1.xlsx"
path_ribosomal_RNA = f"{base}/data/row/Ribosomal-RNA-1.xlsx"

#Viral
path_viral_target = f"{base}/data/row/Viral-target-1.xlsx"
path_viral_annotation = f"{base}/data/row/Viral-annotation-1.xlsx"
path_viral_RNA = f"{base}/data/row/Viral-RNA-1.xlsx"

#Riboswitch
path_riboswitch_target = f"{base}/data/row/Riboswitch-target-1.xlsx"
path_riboswitch_annotation = f"{base}/data/row/Riboswitch-annotation-1.xlsx"
path_riboswitch_RNA = f"{base}/data/row/Riboswitch-RNA-1.xlsx"


In [5]:
#miRNA
miRNA_target = pd.read_excel(path_miRNA_target, engine="openpyxl")
miRNA_annotation = pd.read_excel(path_miRNA_annotation, engine="openpyxl")
miRNA_RNA = pd.read_excel(path_miRNA_RNA, engine="openpyxl")

#Repeats
repeats_target = pd.read_excel(path_repeats_target, engine="openpyxl")
repeats_annotation = pd.read_excel(path_repeats_annotation, engine="openpyxl")
repeats_RNA = pd.read_excel(path_repeats_RNA, engine="openpyxl")

#Ribosomal
ribosomal_target = pd.read_excel(path_ribosomal_target, engine="openpyxl")
ribosomal_annotation = pd.read_excel(path_ribosomal_annotation, engine="openpyxl")
ribosomal_RNA = pd.read_excel(path_ribosomal_RNA, engine="openpyxl")

#Viral
viral_target = pd.read_excel(path_viral_target, engine="openpyxl")
viral_annotation = pd.read_excel(path_viral_annotation, engine="openpyxl")
viral_RNA = pd.read_excel(path_viral_RNA, engine="openpyxl")

#Riboswitch
riboswitch_target = pd.read_excel(path_riboswitch_target, engine="openpyxl")
riboswitch_annotation = pd.read_excel(path_riboswitch_annotation, engine="openpyxl")
riboswitch_RNA = pd.read_excel(path_riboswitch_RNA, engine="openpyxl")


In [253]:
miRNA_RNA = miRNA_annotation[['RNA_name', 'rna_content']]
miRNA_RNA.shape
miRNA_target = miRNA_annotation[['small_molecule_name', 'small_molecule_content']]
miRNA_target.shape

a = (
    miRNA_RNA
        [["RNA_name", "rna_content"]]
        .rename(columns={
            "RNA_name": "name",
            "rna_content": "content"
        })
)

b = (
    miRNA_target
        [["small_molecule_name", "small_molecule_content"]]
        .rename(columns={
            "small_molecule_name": "name",
            "small_molecule_content": "content"
        })
)

miRNA = pd.concat([a, b])

In [ ]:
miRNA.dropna(inplace=True)
miRNA.shape
res = miRNA["content"].apply(classify_sequence)

kinds = res.apply(lambda x: x.kind).value_counts()
kinds

content
SMILES     146
RNA        133
INVALID     12
DNA          1
Name: count, dtype: int64

In [260]:
mask_INVALID = res.apply(lambda x: x.kind) == "INVALID"
INVALID_rows = miRNA.loc[mask_INVALID, ["content", "name"]].copy()
INVALID_rows["invalid_chars"] = res[mask_INVALID].apply(lambda x: x.invalid_chars)
INVALID_rows["len"] = res[mask_INVALID].apply(lambda x: x.length)
INVALID_rows = INVALID_rows.reset_index()
INVALID_rows = INVALID_rows[["index", "content", "name", "invalid_chars", "len"]]
INVALID_rows.to_csv("miRNA_INVALID.csv")
INVALID_rows

,index,content,name,invalid_chars,len
0,98,CCGACUGAUGUUGAXUGUUGAAUYUCAUGGCAACACCAGUCGG,Oligonucleotide 4,XY,43
1,99,CCGACUGAUGUUGAXUGUUGAAUYUCAUGGCAACACCAGUCGG,Oligonucleotide 4,XY,43
2,100,CCGACUGAUGUUGAXUGUUGAAUYUCAUGGCAACACCAGUCGG,Oligonucleotide 4,XY,43
3,101,CCGACUGAUGUUGAXUGUUGAAUYUCAUGGCAACACCAGUCGG,Oligonucleotide 4,XY,43
4,102,CCGACUGAUGUUGAXUGUUGAAUYUCAUGGCAACACCAGUCGG,Oligonucleotide 4,XY,43
5,103,CCGACUGAUGUUGAXUGUUGAAUYUCAUGGCAACACCAGUCGG,Oligonucleotide 4,XY,43
6,104,CCGACUGAUGUUGAXUGUUGAAUYUCAUGGCAACACCAGUCGG,Oligonucleotide 4,XY,43
7,105,CCGACUGAUGUUGAXUGUUGAAUYUCAUGGCAACACCAGUCGG,Oligonucleotide 4,XY,43
8,106,CCGACUGAUGUUGAXUGUUGAAUYUCAUGGCAACACCAGUCGG,Oligonucleotide 4,XY,43
9,107,CCGACUGAUGUUGAXUGUUGAAUYUCAUGGCAACACCAGUCGG,Oligonucleotide 4,XY,43


In [256]:
INVALID_rows.iloc[-1]["content"]

'Coc1ccc(cc1OCC1=NC2N(C1)C=CC=C2)NC(=O)CCCOc1ccc(cc1)c1[nH]c2c(n1)ccc(c2)c1ccc(cc1)N1CCN(CC1)C'

In [257]:
old = 'Coc1ccc(cc1OCC1=NC2N(C1)C=CC=C2)NC(=O)CCCOc1ccc(cc1)c1[nH]c2c(n1)ccc(c2)c1ccc(cc1)N1CCN(CC1)C'
new = 'COc1ccc(cc1OCC1=NC2N(C1)C=CC=C2)NC(=O)CCCOc1ccc(cc1)c1[nH]c2c(n1)ccc(c2)c1ccc(cc1)N1CCN(CC1)C'

mask = miRNA["content"].eq(old)

miRNA.loc[mask, "content"] = new

In [261]:
mask_SMILES = res.apply(lambda x: x.kind) == "SMILES"
SMILES_rows = miRNA.loc[mask_SMILES, ["content", "name"]].copy()
SMILES_rows["invalid_chars"] = res[mask_SMILES].apply(lambda x: x.invalid_chars)
SMILES_rows["len"] = res[mask_SMILES].apply(lambda x: x.length)
SMILES_rows = SMILES_rows.reset_index()
SMILES_rows = SMILES_rows[["index", "content", "name", "invalid_chars", "len"]]
SMILES_rows.to_csv("miRNA_SMILES.csv")
SMILES_rows

,index,content,name,invalid_chars,len
0,0,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...,Targaprimir-96,,199
1,1,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...,Targaprimir-96,,199
2,2,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...,Targaprimir-96,,199
3,3,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...,Targaprimir-96,,199
4,4,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...,Targaprimir-96,,199
...,...,...,...,...,...
141,141,CCCN(C(=O)CN(C(=O)NCc1nnn(c1)CCCCC(=O)CCCOc1cc...,SMB3,,200
142,142,CCCN(C(=O)CN(C(=O)NCc1nnn(c1)CCCCC(=O)CCCOc1c(...,SMB4,,236
143,143,OC(COP(=O)(OC1C(O)C(OC1n1cnc2c1ncnc2N)COP(=O)(...,SMC1,,239
144,144,CCCN(C(=O)N(Cc1nnn(c1)CCCCC(=O)CCCOc1c(cc(cc1C...,SMC2,,324


In [262]:
mask_RNA = res.apply(lambda x: x.kind) == "RNA"
RNA_rows = miRNA.loc[mask_RNA, ["content", "name"]].copy()
RNA_rows["invalid_chars"] = res[mask_RNA].apply(lambda x: x.invalid_chars)
RNA_rows["len"] = res[mask_RNA].apply(lambda x: x.length)
RNA_rows = RNA_rows.reset_index()
RNA_rows = RNA_rows[["index", "content", "name", "invalid_chars", "len"]]
RNA_rows.to_csv("miRNA_RNA.csv")
RNA_rows

,index,content,name,invalid_chars,len
0,0,GGGAGAGGGUUUAAUUACGAAAGUAAUUGGAUCCGCAAGG,Pri-miR-96 RNAC (UU loop mutant to AU basepair),,40
1,1,GGGAGAGGGUUUAUUUACGAAAGUAUAUGGAUCCGCAAGG,Pri-miR-96 RNA1 (Drosha processing site only),,40
2,2,GGGAGAGGGUUUACGAACGAAAGUUGGUGGAUCCGCAAGG,Pri-miR-96 RNA2 (desired GG loop only),,40
3,3,GGGAGAGGGUUUACGAUUUACGAAAGUAUAUGGUGGAUCCGCAAGG,Pri-miR-96 RNA3 (both the Drosha target site a...,,46
4,4,GGGAGAGGGUUUACGAUUUACGAAAGUAUAUCGUGGAUCCGCAAGG,Pri-miR-96 RNA4 (GG to GC loop mutation),,46
...,...,...,...,...,...
128,141,AGCCCCUGCCCACCGCACACUGCGCUGCCCCAGACCCACUGUGCGU...,Pre-miR-21,,60
129,142,UAGCUUAUCAGACUGAUGUUGACUGUUGAAUCUCAUGGCAACACCA...,Pre-miR-17/18a/20a,,59
130,143,AGCCCCUGCCCACCGCACACUGCGCUGCCCCAGACCCACUGUGCGU...,Pre-miR-210,,60
131,144,GGGAGAGGGUUUAAUUACGAAAGUAAUUGGAUCCGCAAGG,Pri-miR-96,,40


In [263]:
mask_DNA = res.apply(lambda x: x.kind) == "DNA"
DNA_rows = miRNA.loc[mask_DNA, ["content", "name"]].copy()
DNA_rows["invalid_chars"] = res[mask_DNA].apply(lambda x: x.invalid_chars)
DNA_rows["len"] = res[mask_DNA].apply(lambda x: x.length)
DNA_rows = DNA_rows.reset_index()
DNA_rows = DNA_rows[["index", "content", "name", "invalid_chars", "len"]]
DNA_rows.to_csv("miRNA_DNA.csv")
DNA_rows

,index,content,name,invalid_chars,len
0,6,CGCGAATTCGCGTTTTCGCGAATTCGCG,Pri-miR-96 AT-rich DNA hairpin,,28


In [264]:
repeats_annotation

,RNA_name,rna_content,small_molecule_name,small_molecule_content,kd
0,r(CUG),GCGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCGC,hexanamide derivative_1,NCCCC[C@@H](NC(=O)[C@@H](NC(=O)[C@H]1CCCN1C(=O...,5.154902
1,r(CUG),GCGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCGC,hexanamide derivative_3,CCC1=NC2=CC=CC=C2C=C1C(=O)N3CCC[C@H]3C(=O)N[C@...,5.000000
2,r(CUG),GCGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCGC,butanediamide derivative_1,CCc1nc2c(cccc2)cc1C(=O)N1CCC[C@@H]1C(=O)N[C@H]...,5.259637
3,r(CUG),GCGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCGC,hexanamide derivative_2,CCc1nc2c(cccc2)cc1C(=O)N1CCC[C@@H]1C(=O)N[C@H]...,5.124939
4,r(CUG),GCGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCGC,"triazine-2,4,6-triamine derivative",COC1=CC2=C(C3=C(C=C(C=C3)Cl)N=C2C=C1)NCCCCNC4=...,4.958607
...,...,...,...,...,...
92,r(CGGx60),UUGGGCCGGCGGCGGCGGCGGCGGCGGCGGCGGCGGCGGCGGCGGC...,Curcumin,COC1=C(C=CC(=C1)C=CC(=O)CC(=O)C=CC2=CC(=C(C=C2...,7.886057
93,r(AUx6),AUAUAUAUAUAU,Curcumin,COC1=C(C=CC(=C1)C=CC(=O)CC(=O)C=CC2=CC(=C(C=C2...,5.636388
94,r(CAGx6),UUGGGCCAGCAGCAGCAGCAGCAGGUCC,Curcumin,COC1=C(C=CC(=C1)C=CC(=O)CC(=O)C=CC2=CC(=C(C=C2...,5.795880
95,r(CCGx6),UUGGGCCCGCCGCCGCCGCCGCCGGUCC,Curcumin,COC1=C(C=CC(=C1)C=CC(=O)CC(=O)C=CC2=CC(=C(C=C2...,6.075721


In [265]:
repeats_target = repeats_annotation[['small_molecule_name', 'small_molecule_content']]
repeats_target.shape
repeats_RNA = repeats_annotation[['RNA_name', 'rna_content']]
repeats_RNA.shape

a = (
    repeats_annotation
        [["RNA_name", "rna_content"]]
        .rename(columns={
            "RNA_name": "name",
            "rna_content": "content"
        })
)

b = (
    repeats_annotation
        [["small_molecule_name", "small_molecule_content"]]
        .rename(columns={
            "small_molecule_name": "name",
            "small_molecule_content": "content"
        })
)

repeats = pd.concat([a, b])

In [193]:
repeats

,name,content
0,r(CUG),GCGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCGC
1,r(CUG),GCGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCGC
2,r(CUG),GCGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCGC
3,r(CUG),GCGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCGC
4,r(CUG),GCGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCGC
...,...,...
92,Curcumin,COC1=C(C=CC(=C1)C=CC(=O)CC(=O)C=CC2=CC(=C(C=C2...
93,Curcumin,COC1=C(C=CC(=C1)C=CC(=O)CC(=O)C=CC2=CC(=C(C=C2...
94,Curcumin,COC1=C(C=CC(=C1)C=CC(=O)CC(=O)C=CC2=CC(=C(C=C2...
95,Curcumin,COC1=C(C=CC(=C1)C=CC(=O)CC(=O)C=CC2=CC(=C(C=C2...


In [ ]:
repeats.shape
res = repeats["content"].apply(classify_sequence)

kinds = res.apply(lambda x: x.kind).value_counts()
kinds

content
SMILES           97
RNA              74
NUCLEIC_AMBIG    22
DNA               1
Name: count, dtype: int64

In [267]:
mask_SMILES = res.apply(lambda x: x.kind) == "SMILES"
SMILES_rows = repeats.loc[mask_SMILES, ["content", "name"]].copy()
SMILES_rows["invalid_chars"] = res[mask_SMILES].apply(lambda x: x.invalid_chars)
SMILES_rows["len"] = res[mask_SMILES].apply(lambda x: x.length)
SMILES_rows = SMILES_rows.reset_index()
SMILES_rows = SMILES_rows[["index", "content", "name", "invalid_chars", "len"]]
SMILES_rows
SMILES_rows.to_csv("repeats_SMILES.csv")

In [268]:
mask_RNA = res.apply(lambda x: x.kind) == "RNA"
RNA_rows = repeats.loc[mask_RNA, ["content", "name"]].copy()
RNA_rows["invalid_chars"] = res[mask_RNA].apply(lambda x: x.invalid_chars)
RNA_rows["len"] = res[mask_RNA].apply(lambda x: x.length)
RNA_rows = RNA_rows.reset_index()
RNA_rows = RNA_rows[["index", "content", "name", "invalid_chars", "len"]]
RNA_rows
RNA_rows.to_csv("repeats_RNA.csv")

mask_DNA = res.apply(lambda x: x.kind) == "DNA"
DNA_rows = repeats.loc[mask_DNA, ["content", "name"]].copy()
DNA_rows["invalid_chars"] = res[mask_DNA].apply(lambda x: x.invalid_chars)
DNA_rows["len"] = res[mask_DNA].apply(lambda x: x.length)
DNA_rows = DNA_rows.reset_index()
DNA_rows = DNA_rows[["index", "content", "name", "invalid_chars", "len"]]
DNA_rows
DNA_rows.to_csv("repeats_DNA.csv")

mask_NUCLEIC_AMBIG = res.apply(lambda x: x.kind) == "NUCLEIC_AMBIG"
NUCLEIC_AMBIG_rows = repeats.loc[mask_NUCLEIC_AMBIG, ["content", "name"]].copy()
NUCLEIC_AMBIG_rows["invalid_chars"] = res[mask_NUCLEIC_AMBIG].apply(lambda x: x.invalid_chars)
NUCLEIC_AMBIG_rows["len"] = res[mask_NUCLEIC_AMBIG].apply(lambda x: x.length)
NUCLEIC_AMBIG_rows = NUCLEIC_AMBIG_rows.reset_index()
NUCLEIC_AMBIG_rows = NUCLEIC_AMBIG_rows[["index", "content", "name", "invalid_chars", "len"]]
NUCLEIC_AMBIG_rows
NUCLEIC_AMBIG_rows.to_csv("repeats_NUCLEIC_AMBIG.csv")

In [7]:
ribosomal_annotation.rename(columns={
    "small_moleculecontent": "small_molecule_content"
}, inplace=True)


In [8]:
ribosomal_annotation

,RNA_name,rna_content,small_molecule_name,small_molecule_content,kd
0,A-site 16S rRNA_E_coli (1),GAGCGUCACACCUUCGGGUGAAGUCGCUC,tetracyclin,CC1(C2CC3C(C(=O)C(=C(C3(C(=O)C2=C(C4=C1C=CC=C4...,5.455932
1,A-site 16S rRNA_E_coli (1),GAGCGUCACACCUUCGGGUGAAGUCGCUC,Spectinomycin,CC1CC(=O)C2(C(O1)OC3C(C(C(C(C3O2)NC)O)NC)O)O,6.585027
2,A-site 16S rRNA_E_coli (1),GAGCGUCACACCUUCGGGUGAAGUCGCUC,gentamicin_mol_c1a,CC1(COC(C(C1NC)O)OC2C(CC(C(C2O)OC3C(CCC(O3)CN)...,5.769551
3,A-site 16S rRNA_E_coli (1),GAGCGUCACACCUUCGGGUGAAGUCGCUC,kanamycin,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CN)O)O)O)O)OC3C(C(...,6.903090
4,A-site 16S rRNA_E_coli (1),GAGCGUCACACCUUCGGGUGAAGUCGCUC,Kasugamycin,CC1C(CC(C(O1)OC2C(C(C(C(C2O)O)O)O)O)N)N=C(C(=O...,4.823909
...,...,...,...,...,...
290,16s_rRNA A SITE,GAGCGUCACACCUUCGGGUGAAGUCGCUC,Neamine_derivative_30,N[C@@H]1C[C@H](N)[C@H]([C@@H]([C@H]1OCn1nccc1C...,3.355561
291,16s_rRNA A SITE,GAGCGUCACACCUUCGGGUGAAGUCGCUC,Neamine_derivative_33,N[C@@H]1C[C@H](N)[C@H]([C@@H]([C@H]1OCn1cnc2c1...,4.000000
292,16s_rRNA A SITE,GAGCGUCACACCUUCGGGUGAAGUCGCUC,Apramycin,CNC1C(OC2OC(Cn3nnc(c3)CCC(=O)Nc3cccc(c3)c3csc(...,5.698970
293,16s_rRNA A SITE,GAGCGUCACACCUUCGGGUGAAGUCGCUC,Bekanamycin,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)N)O)O)OC3C(C(...,5.698970


In [9]:
ribosomal_target = ribosomal_annotation[['small_molecule_name', 'small_molecule_content']]
ribosomal_target.shape
ribosomal_RNA = ribosomal_annotation[['RNA_name', 'rna_content']]
ribosomal_RNA.shape

a = (
    ribosomal_annotation
        [["RNA_name", "rna_content"]]
        .rename(columns={
            "RNA_name": "name",
            "rna_content": "content"
        })
)

b = (
    ribosomal_annotation
        [["small_molecule_name", "small_molecule_content"]]
        .rename(columns={
            "small_molecule_name": "name",
            "small_molecule_content": "content"
        })
)

ribosomal = pd.concat([a, b])

In [ ]:
ribosomal.shape
res = ribosomal["content"].apply(classify_sequence)

kinds = res.apply(lambda x: x.kind).value_counts()
kinds

content
SMILES     295
RNA        277
INVALID     18
Name: count, dtype: int64

In [17]:
mask_SMILES = res.apply(lambda x: x.kind) == "SMILES"
SMILES_rows = ribosomal.loc[mask_SMILES, ["content", "name"]].copy()
SMILES_rows["invalid_chars"] = res[mask_SMILES].apply(lambda x: x.invalid_chars)
SMILES_rows["len"] = res[mask_SMILES].apply(lambda x: x.length)
SMILES_rows = SMILES_rows.reset_index()
SMILES_rows = SMILES_rows[["index", "content", "name", "invalid_chars", "len"]]
SMILES_rows.to_csv("ribosomal_SMILES.csv")
SMILES_rows

,index,content,name,invalid_chars,len
0,0,CC1(C2CC3C(C(=O)C(=C(C3(C(=O)C2=C(C4=C1C=CC=C4...,tetracyclin,,68
1,1,CC1CC(=O)C2(C(O1)OC3C(C(C(C(C3O2)NC)O)NC)O)O,Spectinomycin,,44
2,2,CC1(COC(C(C1NC)O)OC2C(CC(C(C2O)OC3C(CCC(O3)CN)...,gentamicin_mol_c1a,,53
3,3,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CN)O)O)O)O)OC3C(C(...,kanamycin,,63
4,4,CC1C(CC(C(O1)OC2C(C(C(C(C2O)O)O)O)O)N)N=C(C(=O...,Kasugamycin,,50
...,...,...,...,...,...
290,290,N[C@@H]1C[C@H](N)[C@H]([C@@H]([C@H]1OCn1nccc1C...,Neamine_derivative_30,,57
291,291,N[C@@H]1C[C@H](N)[C@H]([C@@H]([C@H]1OCn1cnc2c1...,Neamine_derivative_33,,60
292,292,CNC1C(OC2OC(Cn3nnc(c3)CCC(=O)Nc3cccc(c3)c3csc(...,Apramycin,,107
293,293,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)N)O)O)OC3C(C(...,Bekanamycin,,63


In [11]:
mask_INVALID = res.apply(lambda x: x.kind) == "INVALID"
INVALID_rows = ribosomal.loc[mask_INVALID, ["content", "name"]].copy()
INVALID_rows["invalid_chars"] = res[mask_INVALID].apply(lambda x: x.invalid_chars)
INVALID_rows["len"] = res[mask_INVALID].apply(lambda x: x.length)
INVALID_rows = INVALID_rows.reset_index()
INVALID_rows = INVALID_rows[["index", "content", "name", "invalid_chars", "len"]]
INVALID_rows.to_csv("ribosomal_INVALID.csv")
INVALID_rows

,index,content,name,invalid_chars,len
0,78,GGUUAAGCGACUAAGCGUACACGGUGGAUGCCCUGGCAGUCAGAGG...,50S subunit,X,2897
1,79,GGUUAAGCGACUAAGCGUACACGGUGGAUGCCCUGGCAGUCAGAGG...,50S subunit,X,2897
2,156,GGUUAAGCGACUAAGCGUACACGGUGGAUGCCCUGGCAGUCAGAGG...,50S A-site for puromycin,X,2897
3,157,GGUUAAGCGACUAAGCGUACACGGUGGAUGCCCUGGCAGUCAGAGG...,large ribosomal subunit from Deinococcus radio...,X,2897
4,158,GGUUAAGCGACUAAGCGUACACGGUGGAUGCCCUGGCAGUCAGAGG...,50S exit tunnel for SPIRAMYCIN A,X,2897
5,159,GGUUAAGCGACUAAGCGUACACGGUGGAUGCCCUGGCAGUCAGAGG...,Large subunit (TELITHROMYCIN),X,2897
6,162,GGUUAAGCGACUAAGCGUACACGGUGGAUGCCCUGGCAGUCAGAGG...,Exit tunnel for TYLOSIN,X,2897
7,163,GGUUAAGCGACUAAGCGUACACGGUGGAUGCCCUGGCAGUCAGAGG...,50S ribosome (CLINDAMYCIN),X,2897
8,164,GGUUAAGCGACUAAGCGUACACGGUGGAUGCCCUGGCAGUCAGAGG...,50S ribosome (Virginiamycin M),X,2897
9,170,GGUUAAGCGACUAAGCGUACACGGUGGAUGCCCUGGCAGUCAGAGG...,50S SB 280080,X,2897


In [12]:
INVALID_rows.iloc[-1]['content'], INVALID_rows.iloc[-2]['content'], INVALID_rows.iloc[-3]['content']

('c1(c(cc2c(c1)nc(n2Cc1ccc(cc1N(=O)O)N(=O)O)C1CCNCC1)Cl)Cl',
 'c1(c(cc2c(c1)nc(n2Cc1ccc(cc1N(=O)O)C(F)(F)F)C1CCNCC1)Cl)Cl',
 'c1c(cc2c(c1)[nH]c(n2)[C@H]1C[C@H](CCC1)N)N(=O)O')

In [ ]:
old = 'c1(c(cc2c(c1)nc(n2Cc1ccc(cc1N(=O)O)N(=O)O)C1CCNCC1)Cl)Cl'
new = 'O=[N+]([O-])c1ccc(Cn2c(C3CCNCC3)nc3cc(Cl)c(Cl)cc32)c([N+](=O)[O-])c1'

mask = ribosomal["content"].eq(old)

ribosomal.loc[mask, "content"] = new

In [ ]:
old = 'c1(c(cc2c(c1)nc(n2Cc1ccc(cc1N(=O)O)C(F)(F)F)C1CCNCC1)Cl)Cl'
new = 'O=[N+]([O-])c1cc(C(F)(F)F)ccc1Cn1c(C2CCNCC2)nc2cc(Cl)c(Cl)cc21'

mask = ribosomal["content"].eq(old)

ribosomal.loc[mask, "content"] = new

In [15]:
old = 'c1c(cc2c(c1)[nH]c(n2)[C@H]1C[C@H](CCC1)N)N(=O)O'
new = 'N[C@H]1CCC[C@@H](c2nc3cc([N+](=O)[O-])ccc3[nH]2)C1'

mask = ribosomal["content"].eq(old)

ribosomal.loc[mask, "content"] = new

In [281]:
mask_RNA = res.apply(lambda x: x.kind) == "RNA"
RNA_rows = ribosomal.loc[mask_RNA, ["content", "name"]].copy()
RNA_rows["invalid_chars"] = res[mask_RNA].apply(lambda x: x.invalid_chars)
RNA_rows["len"] = res[mask_RNA].apply(lambda x: x.length)
RNA_rows = RNA_rows.reset_index()
RNA_rows = RNA_rows[["index", "content", "name", "invalid_chars", "len"]]
RNA_rows.to_csv("ribosomal_RNA.csv")
RNA_rows

,index,content,name,invalid_chars,len
0,0,GAGCGUCACACCUUCGGGUGAAGUCGCUC,A-site 16S rRNA_E_coli (1),,29
1,1,GAGCGUCACACCUUCGGGUGAAGUCGCUC,A-site 16S rRNA_E_coli (1),,29
2,2,GAGCGUCACACCUUCGGGUGAAGUCGCUC,A-site 16S rRNA_E_coli (1),,29
3,3,GAGCGUCACACCUUCGGGUGAAGUCGCUC,A-site 16S rRNA_E_coli (1),,29
4,4,GAGCGUCACACCUUCGGGUGAAGUCGCUC,A-site 16S rRNA_E_coli (1),,29
...,...,...,...,...,...
272,290,GAGCGUCACACCUUCGGGUGAAGUCGCUC,16s_rRNA A SITE,,29
273,291,GAGCGUCACACCUUCGGGUGAAGUCGCUC,16s_rRNA A SITE,,29
274,292,GAGCGUCACACCUUCGGGUGAAGUCGCUC,16s_rRNA A SITE,,29
275,293,GAGCGUCACACCUUCGGGUGAAGUCGCUC,16s_rRNA A SITE,,29


In [282]:
viral_annotation

,RNA_name,rna_content,small_molecule_name,small_molecule_content,kd
0,RNA_Str_HIV-2 TAR,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,neocarzinostatin,CC1C(C(C(C(O1)OC2C3CC#CC4C(O4)(C#CC3=CC2OC(=O)...,4.853872
1,RNA_Str_HIV-2 TAR,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,Neocarzinostatin_chromophore_derivative_1,c12c(cc(cc1C)OC)[C@@]1(C(=O)C=C2)C(=O)O[C@@H]2...,4.853872
2,RNA_Str_HIV-2 TAR,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,Neocarzinostatin_chromophore_derivative_2,c12c(cccc1)[C@]1(C(=O)C=C2)[C@@H](O[C@@H]2[C@@...,4.853872
3,RNA_Str_HIV-2 TAR,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,Neocarzinostatin_chromophore_derivative_3,c12c(cccc1)[C@]1([C@@H](C=C2)O[C@H]2[C@@H]([C@...,4.886057
4,RNA_Str_HIV-2 TAR,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,Neocarzinostatin_chromophore_derivative_5,c12c(cc(cc1)OC)[C@@]1(C(=O)C=C2)C=C[C@@H]2[C@@...,5.096910
...,...,...,...,...,...
276,RRE4,GGUGGGCGCAGCUUCGGCUGCGGACACC,Paromomycin,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)O)N)OC3C(C(C(...,4.809668
277,RRE11,GGUGGGCGCAGCUUCGGCUGCGCAACCACC,Paromomycin,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)O)N)OC3C(C(C(...,4.728158
278,RRE3,GGUGGGCGCAGCUUCGGCUGCGCCCACC,Kanamycin B,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)N)O)O)OC3C(C(...,4.903090
279,RRE4,GGUGGGCGCAGCUUCGGCUGCGGACACC,Kanamycin B,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)N)O)O)OC3C(C(...,5.200659


In [283]:
viral_target = repeats_annotation[['small_molecule_name', 'small_molecule_content']]
viral_target.shape
viral_RNA = repeats_annotation[['RNA_name', 'rna_content']]
viral_RNA.shape

a = (
    viral_annotation
        [["RNA_name", "rna_content"]]
        .rename(columns={
            "RNA_name": "name",
            "rna_content": "content"
        })
)

b = (
    viral_annotation
        [["small_molecule_name", "small_molecule_content"]]
        .rename(columns={
            "small_molecule_name": "name",
            "small_molecule_content": "content"
        })
)

viral = pd.concat([a, b])

In [ ]:
viral.shape
res = viral["content"].apply(classify_sequence)

kinds = res.apply(lambda x: x.kind).value_counts()
kinds

content
SMILES           281
RNA              279
NUCLEIC_AMBIG      2
Name: count, dtype: int64

In [292]:
mask_SMILES = res.apply(lambda x: x.kind) == "SMILES"
SMILES_rows = viral.loc[mask_SMILES, ["content", "name"]].copy()
SMILES_rows["invalid_chars"] = res[mask_SMILES].apply(lambda x: x.invalid_chars)
SMILES_rows["len"] = res[mask_SMILES].apply(lambda x: x.length)
SMILES_rows = SMILES_rows.reset_index()
SMILES_rows = SMILES_rows[["index", "content", "name", "invalid_chars", "len"]]
SMILES_rows.to_csv("viral_SMILES.csv")
SMILES_rows

,index,content,name,invalid_chars,len
0,0,CC1C(C(C(C(O1)OC2C3CC#CC4C(O4)(C#CC3=CC2OC(=O)...,neocarzinostatin,,92
1,1,c12c(cc(cc1C)OC)[C@@]1(C(=O)C=C2)C(=O)O[C@@H]2...,Neocarzinostatin_chromophore_derivative_1,,144
2,2,c12c(cccc1)[C@]1(C(=O)C=C2)[C@@H](O[C@@H]2[C@@...,Neocarzinostatin_chromophore_derivative_2,,116
3,3,c12c(cccc1)[C@]1([C@@H](C=C2)O[C@H]2[C@@H]([C@...,Neocarzinostatin_chromophore_derivative_3,,116
4,4,c12c(cc(cc1)OC)[C@@]1(C(=O)C=C2)C=C[C@@H]2[C@@...,Neocarzinostatin_chromophore_derivative_5,,119
...,...,...,...,...,...
276,276,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)O)N)OC3C(C(C(...,Paromomycin,,80
277,277,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)O)N)OC3C(C(C(...,Paromomycin,,80
278,278,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)N)O)O)OC3C(C(...,Kanamycin B,,63
279,279,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)N)O)O)OC3C(C(...,Kanamycin B,,63


In [293]:
mask_RNA = res.apply(lambda x: x.kind) == "RNA"
RNA_rows = viral.loc[mask_RNA, ["content", "name"]].copy()
RNA_rows["invalid_chars"] = res[mask_RNA].apply(lambda x: x.invalid_chars)
RNA_rows["len"] = res[mask_RNA].apply(lambda x: x.length)
RNA_rows = RNA_rows.reset_index()
RNA_rows = RNA_rows[["index", "content", "name", "invalid_chars", "len"]]
RNA_rows.to_csv("viral_RNA.csv")
RNA_rows

,index,content,name,invalid_chars,len
0,0,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,RNA_Str_HIV-2 TAR,,30
1,1,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,RNA_Str_HIV-2 TAR,,30
2,2,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,RNA_Str_HIV-2 TAR,,30
3,3,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,RNA_Str_HIV-2 TAR,,30
4,4,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,RNA_Str_HIV-2 TAR,,30
...,...,...,...,...,...
274,276,GGUGGGCGCAGCUUCGGCUGCGGACACC,RRE4,,28
275,277,GGUGGGCGCAGCUUCGGCUGCGCAACCACC,RRE11,,30
276,278,GGUGGGCGCAGCUUCGGCUGCGCCCACC,RRE3,,28
277,279,GGUGGGCGCAGCUUCGGCUGCGGACACC,RRE4,,28


In [294]:
mask_NUCLEIC_AMBIG = res.apply(lambda x: x.kind) == "NUCLEIC_AMBIG"
NUCLEIC_AMBIG_rows = viral.loc[mask_NUCLEIC_AMBIG, ["content", "name"]].copy()
NUCLEIC_AMBIG_rows["invalid_chars"] = res[mask_NUCLEIC_AMBIG].apply(lambda x: x.invalid_chars)
NUCLEIC_AMBIG_rows["len"] = res[mask_NUCLEIC_AMBIG].apply(lambda x: x.length)
NUCLEIC_AMBIG_rows = NUCLEIC_AMBIG_rows.reset_index()
NUCLEIC_AMBIG_rows = NUCLEIC_AMBIG_rows[["index", "content", "name", "invalid_chars", "len"]]
NUCLEIC_AMBIG_rows.to_csv("viral_NUCLEIC_AMBIG.csv")
NUCLEIC_AMBIG_rows

,index,content,name,invalid_chars,len
0,168,GGAGGAGGGGGAGGAGGA,HCV IRES-A,,18
1,172,GGAGGAGGGGGAGGAGGA,IRES RNA G-quadruplex,,18


In [295]:
mask_INVALID = res.apply(lambda x: x.kind) == "INVALID"
INVALID_rows = viral.loc[mask_INVALID, ["content", "name"]].copy()
INVALID_rows["invalid_chars"] = res[mask_INVALID].apply(lambda x: x.invalid_chars)
INVALID_rows["len"] = res[mask_INVALID].apply(lambda x: x.length)
INVALID_rows = INVALID_rows.reset_index()
INVALID_rows = INVALID_rows[["index", "content", "name", "invalid_chars", "len"]]
# INVALID_rows.to_csv("viral_INVALID.csv")
INVALID_rows

,index,content,name,invalid_chars,len


In [287]:
INVALID_rows.iloc[-1]['content'], INVALID_rows.iloc[-2]['content'], INVALID_rows.iloc[-3]['content']

('Coc1ccc2c(c1)c(Nc1ccc(c(c1)CN1CCN(CC1)C)O)c1c(n2)cc(cc1)Cl',
 'Ccc1nc2ccccc2cc1C(=O)O',
 '[C@H]12[C@@H](C[C@@H]3[N@@](C1)(CCc1c3[nH]c3c1cccc3)C)[C@H]([C@H](CC2)O)C(=O)OC')

In [ ]:
old = 'Coc1ccc2c(c1)c(Nc1ccc(c(c1)CN1CCN(CC1)C)O)c1c(n2)cc(cc1)Cl'
new = 'COc1ccc2nc3cc(Cl)ccc3c(Nc3ccc(O)c(CN4CCN(C)CC4)c3)c2c1'

mask = viral["content"].eq(old)

viral.loc[mask, "content"] = new

In [ ]:
old = 'Ccc1nc2ccccc2cc1C(=O)O'
new = 'CCc1nc2ccccc2cc1C(=O)O'

mask = viral["content"].eq(old)

viral.loc[mask, "content"] = new

In [290]:
old = '[C@H]12[C@@H](C[C@@H]3[N@@](C1)(CCc1c3[nH]c3c1cccc3)C)[C@H]([C@H](CC2)O)C(=O)OC'
new = '[C@H]12[C@@H](C[C@@H]3[N@@+](C1)(CCc1c3[nH]c3c1cccc3)C)[C@H]([C@H](CC2)O)C(=O)OC'

mask = viral["content"].eq(old)

viral.loc[mask, "content"] = new

In [296]:
riboswitch_annotation

,RNA_name,rna_content,small_molecule_name,small_molecule_content,kd
0,ADENINE RIBOSWITCH,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,Adenine,C1=NC2=NC=NC(=C2N1)N,6.397940
1,ADENINE RIBOSWITCH,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,2-aminopurine,C1=C2C(=NC(=N1)N)N=CN2,5.789147
2,ADENINE RIBOSWITCH,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,"2,4,6-triaminopyrimidine",C1=C(N=C(N=C1N)N)N,4.698970
3,ADENINE RIBOSWITCH,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,"2,6- diaminopurine",C1=NC2=NC(=NC(=C2N1)N)N,7.698970
4,ADENINE RIBOSWITCH,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,"2,4-diaminopurine",C1=NC(=NC2(C1=NC=N2)N)N,8.397940
...,...,...,...,...,...
95,Tetracycline riboswitch A50U mutant,GGGCCUAAAACAUACCAGAUCGCCACCCGCGCUUUAAUCUGGAGAG...,dox,C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC...,4.978811
96,FMN RIBOSWITCH APTAMER,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,Ribocil,CNC1=NC=C(C=N1)CN2CCC[C@@H](C2)C3=NC(=CC(=O)N3...,7.795880
97,FMN RIBOSWITCH APTAMER,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,Ribocil-A,CNc1ncc(cn1)CN1CCC[C@H](C1)c1nc(O)cc(n1)c1cccs1,5.000000
98,FMN RIBOSWITCH APTAMER,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,Ribocil-B,CNC1=NC=C(C=N1)CN2CCC[C@@H](C2)C3=NC(=CC(=O)N3...,8.180456


In [297]:
riboswitch_target = riboswitch_annotation[['small_molecule_name', 'small_molecule_content']]
riboswitch_target.shape
riboswitch_RNA = riboswitch_annotation[['RNA_name', 'rna_content']]
riboswitch_RNA.shape

a = (
    riboswitch_annotation
        [["RNA_name", "rna_content"]]
        .rename(columns={
            "RNA_name": "name",
            "rna_content": "content"
        })
)

b = (
    riboswitch_annotation
        [["small_molecule_name", "small_molecule_content"]]
        .rename(columns={
            "small_molecule_name": "name",
            "small_molecule_content": "content"
        })
)

riboswitch = pd.concat([a, b])

In [ ]:
riboswitch.shape
res = riboswitch["content"].apply(classify_sequence)

kinds = res.apply(lambda x: x.kind).value_counts()
kinds

content
RNA       100
SMILES    100
Name: count, dtype: int64

In [304]:
mask_INVALID = res.apply(lambda x: x.kind) == "INVALID"
INVALID_rows = riboswitch.loc[mask_INVALID, ["content", "name"]].copy()
INVALID_rows["invalid_chars"] = res[mask_INVALID].apply(lambda x: x.invalid_chars)
INVALID_rows["len"] = res[mask_INVALID].apply(lambda x: x.length)
INVALID_rows = INVALID_rows.reset_index()
INVALID_rows = INVALID_rows[["index", "content", "name", "invalid_chars", "len"]]
# INVALID_rows.to_csv("riboswitch_INVALID.csv")
INVALID_rows

,index,content,name,invalid_chars,len


In [300]:
INVALID_rows.iloc[-1]['content'], INVALID_rows.iloc[-2]['content']

('Ccn1c2ccc(cc2c2c1cccc2)N1CCNCC1',
 'n1c(ncc(c1N)CN1=CC=CC(=C1)CCO[P@@](=O)(O)P(=O)(O)O)C')

In [ ]:
old = 'Ccn1c2ccc(cc2c2c1cccc2)N1CCNCC1'
new = 'CC[n+]1ccc2c(c1)ccc3c(N4CCNCC4)cccc23'

mask = riboswitch["content"].eq(old)

riboswitch.loc[mask, "content"] = new

In [302]:
old = 'n1c(ncc(c1N)CN1=CC=CC(=C1)CCO[P@@](=O)(O)P(=O)(O)O)C'
new = 'n1c(ncc(c1N)C[n+]1=CC=CC(=C1)CCOP(=O)(O)P(=O)(O)O)C'

mask = riboswitch["content"].eq(old)

riboswitch.loc[mask, "content"] = new

In [305]:
mask_SMILES = res.apply(lambda x: x.kind) == "SMILES"
SMILES_rows = riboswitch.loc[mask_SMILES, ["content", "name"]].copy()
SMILES_rows["invalid_chars"] = res[mask_SMILES].apply(lambda x: x.invalid_chars)
SMILES_rows["len"] = res[mask_SMILES].apply(lambda x: x.length)
SMILES_rows = SMILES_rows.reset_index()
SMILES_rows = SMILES_rows[["index", "content", "name", "invalid_chars", "len"]]
SMILES_rows.to_csv("riboswitch_SMILES.csv")
SMILES_rows

,index,content,name,invalid_chars,len
0,0,C1=NC2=NC=NC(=C2N1)N,Adenine,,20
1,1,C1=C2C(=NC(=N1)N)N=CN2,2-aminopurine,,22
2,2,C1=C(N=C(N=C1N)N)N,"2,4,6-triaminopyrimidine",,18
3,3,C1=NC2=NC(=NC(=C2N1)N)N,"2,6- diaminopurine",,23
4,4,C1=NC(=NC2(C1=NC=N2)N)N,"2,4-diaminopurine",,23
...,...,...,...,...,...
95,95,C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC...,dox,,105
96,96,CNC1=NC=C(C=N1)CN2CCC[C@@H](C2)C3=NC(=CC(=O)N3...,Ribocil,,56
97,97,CNc1ncc(cn1)CN1CCC[C@H](C1)c1nc(O)cc(n1)c1cccs1,Ribocil-A,,47
98,98,CNC1=NC=C(C=N1)CN2CCC[C@@H](C2)C3=NC(=CC(=O)N3...,Ribocil-B,,56


In [307]:
mask_RNA = res.apply(lambda x: x.kind) == "RNA"
RNA_rows = riboswitch.loc[mask_RNA, ["content", "name"]].copy()
RNA_rows["invalid_chars"] = res[mask_RNA].apply(lambda x: x.invalid_chars)
RNA_rows["len"] = res[mask_RNA].apply(lambda x: x.length)
RNA_rows = RNA_rows.reset_index()
RNA_rows = RNA_rows[["index", "content", "name", "invalid_chars", "len"]]
RNA_rows.to_csv("riboswitch_RNA.csv")
RNA_rows

,index,content,name,invalid_chars,len
0,0,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,ADENINE RIBOSWITCH,,67
1,1,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,ADENINE RIBOSWITCH,,67
2,2,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,ADENINE RIBOSWITCH,,67
3,3,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,ADENINE RIBOSWITCH,,67
4,4,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,ADENINE RIBOSWITCH,,67
...,...,...,...,...,...
95,95,GGGCCUAAAACAUACCAGAUCGCCACCCGCGCUUUAAUCUGGAGAG...,Tetracycline riboswitch A50U mutant,,71
96,96,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,FMN RIBOSWITCH APTAMER,,35
97,97,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,FMN RIBOSWITCH APTAMER,,35
98,98,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,FMN RIBOSWITCH APTAMER,,35


In [7]:
miRNA_annotation.drop(columns=['kd'], inplace=True)

In [8]:
miRNA_annotation

,RNA_name,rna_content,small_molecule_name,small_molecule_content
0,Pri-miR-96 RNAC (UU loop mutant to AU basepair),GGGAGAGGGUUUAAUUACGAAAGUAAUUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...
1,Pri-miR-96 RNA1 (Drosha processing site only),GGGAGAGGGUUUAUUUACGAAAGUAUAUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...
2,Pri-miR-96 RNA2 (desired GG loop only),GGGAGAGGGUUUACGAACGAAAGUUGGUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...
3,Pri-miR-96 RNA3 (both the Drosha target site a...,GGGAGAGGGUUUACGAUUUACGAAAGUAUAUGGUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...
4,Pri-miR-96 RNA4 (GG to GC loop mutation),GGGAGAGGGUUUACGAUUUACGAAAGUAUAUCGUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...
...,...,...,...,...
141,Pre-miR-21,AGCCCCUGCCCACCGCACACUGCGCUGCCCCAGACCCACUGUGCGU...,SMB3,CCCN(C(=O)CN(C(=O)NCc1nnn(c1)CCCCC(=O)CCCOc1cc...
142,Pre-miR-17/18a/20a,UAGCUUAUCAGACUGAUGUUGACUGUUGAAUCUCAUGGCAACACCA...,SMB4,CCCN(C(=O)CN(C(=O)NCc1nnn(c1)CCCCC(=O)CCCOc1c(...
143,Pre-miR-210,AGCCCCUGCCCACCGCACACUGCGCUGCCCCAGACCCACUGUGCGU...,SMC1,OC(COP(=O)(OC1C(O)C(OC1n1cnc2c1ncnc2N)COP(=O)(...
144,Pri-miR-96,GGGAGAGGGUUUAAUUACGAAAGUAAUUGGAUCCGCAAGG,SMC2,CCCN(C(=O)N(Cc1nnn(c1)CCCCC(=O)CCCOc1c(cc(cc1C...


In [ ]:
def try_canonical_smiles(s: str):
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

In [9]:
miRNA_annotation['check_rna_seq'] = miRNA_annotation['rna_content'].map(set)
miRNA_annotation['check_smiles_seq'] = miRNA_annotation['small_molecule_content'].map(set)

In [10]:
miRNA_annotation['check_rna_seq'].value_counts()

check_rna_seq
{G, C, U, A}          133
{Y, X, C, U, G, A}     12
{T, A, G, C}            1
Name: count, dtype: int64

In [ ]:
miRNA_annotation['check_smiles_seq'].value_counts()

In [39]:
ALLOWED = set("ACGTU")

s = (
    miRNA_annotation["rna_content"]
    .astype(str)
    .str.strip()
    .str.upper()
)

mask_canonical = s.map(lambda x: set(x).issubset(ALLOWED)) & s.ne("")

nuc_ok = miRNA_annotation[mask_canonical].copy()
nuc_invalid = miRNA_annotation[~mask_canonical].copy()

In [40]:
nuc_ok

,RNA_name,rna_content,small_molecule_name,small_molecule_content,check_rna_seq,check_smiles_seq
0,Pri-miR-96 RNAC (UU loop mutant to AU basepair),GGGAGAGGGUUUAAUUACGAAAGUAAUUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...,"{G, C, U, A}","{(, ), [, ], 1, C, H, +, =, 3, 2, O, N}"
1,Pri-miR-96 RNA1 (Drosha processing site only),GGGAGAGGGUUUAUUUACGAAAGUAUAUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...,"{G, C, U, A}","{(, ), [, ], 1, C, H, +, =, 3, 2, O, N}"
2,Pri-miR-96 RNA2 (desired GG loop only),GGGAGAGGGUUUACGAACGAAAGUUGGUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...,"{G, C, U, A}","{(, ), [, ], 1, C, H, +, =, 3, 2, O, N}"
3,Pri-miR-96 RNA3 (both the Drosha target site a...,GGGAGAGGGUUUACGAUUUACGAAAGUAUAUGGUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...,"{G, C, U, A}","{(, ), [, ], 1, C, H, +, =, 3, 2, O, N}"
4,Pri-miR-96 RNA4 (GG to GC loop mutation),GGGAGAGGGUUUACGAUUUACGAAAGUAUAUCGUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)CC1=CN(CCCNC(...,"{G, C, U, A}","{(, ), [, ], 1, C, H, +, =, 3, 2, O, N}"
...,...,...,...,...,...,...
141,Pre-miR-21,AGCCCCUGCCCACCGCACACUGCGCUGCCCCAGACCCACUGUGCGU...,SMB3,CCCN(C(=O)CN(C(=O)NCc1nnn(c1)CCCCC(=O)CCCOc1cc...,"{G, C, U, A}","{(, ), [, ], 1, C, H, c, =, n, 2, O, N}"
142,Pre-miR-17/18a/20a,UAGCUUAUCAGACUGAUGUUGACUGUUGAAUCUCAUGGCAACACCA...,SMB4,CCCN(C(=O)CN(C(=O)NCc1nnn(c1)CCCCC(=O)CCCOc1c(...,"{G, C, U, A}","{(, ), [, ], 1, C, H, c, =, n, 2, O, N}"
143,Pre-miR-210,AGCCCCUGCCCACCGCACACUGCGCUGCCCCAGACCCACUGUGCGU...,SMC1,OC(COP(=O)(OC1C(O)C(OC1n1cnc2c1ncnc2N)COP(=O)(...,"{G, C, U, A}","{(, ), [, ], 1, H, N, c, P, =, n, 2, O, C}"
144,Pri-miR-96,GGGAGAGGGUUUAAUUACGAAAGUAAUUGGAUCCGCAAGG,SMC2,CCCN(C(=O)N(Cc1nnn(c1)CCCCC(=O)CCCOc1c(cc(cc1C...,"{G, C, U, A}","{(, ), [, ], 1, C, H, c, =, P, n, 2, O, N}"


In [41]:
nuc_ok['canonical_smiles'] = nuc_ok['small_molecule_content'].map(try_canonical_smiles)

[13:46:55] non-ring atom 1 marked aromatic


In [42]:
n_total = len(nuc_ok)
n_valid = nuc_ok['canonical_smiles'].notna().sum()
n_invalid = nuc_ok['canonical_smiles'].isna().sum()

print(f"Total: {n_total}")
print(f"Valid: {n_valid}")
print(f"Invalid: {n_invalid}")

Total: 134
Valid: 133
Invalid: 1


In [43]:
nuc_ok[nuc_ok['canonical_smiles'].isna()]

,RNA_name,rna_content,small_molecule_name,small_molecule_content,check_rna_seq,check_smiles_seq,canonical_smiles
113,Pre-miR-21,AGCCCCUGCCCACCGCACACUGCGCUGCCCCAGACCCACUGUGCGU...,Compound8,Coc1ccc(cc1OCC1=NC2N(C1)C=CC=C2)NC(=O)CCCOc1cc...,"{G, C, U, A}","{(, ), [, ], 1, H, N, c, =, o, n, 2, O, C}",None


In [44]:
old = 'Coc1ccc(cc1OCC1=NC2N(C1)C=CC=C2)NC(=O)CCCOc1ccc(cc1)c1[nH]c2c(n1)ccc(c2)c1ccc(cc1)N1CCN(CC1)C'
new = 'COc1ccc(cc1OCC1=NC2N(C1)C=CC=C2)NC(=O)CCCOc1ccc(cc1)c1[nH]c2c(n1)ccc(c2)c1ccc(cc1)N1CCN(CC1)C'

mask = nuc_ok["small_molecule_content"].eq(old)

nuc_ok.loc[mask, "small_molecule_content"] = new

In [45]:
nuc_ok.drop(columns=['check_rna_seq', 'check_smiles_seq', 'small_molecule_content'], inplace=True)
nuc_ok

,RNA_name,rna_content,small_molecule_name,canonical_smiles
0,Pri-miR-96 RNAC (UU loop mutant to AU basepair),GGGAGAGGGUUUAAUUACGAAAGUAAUUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)Cc1cn(CCCNC(=...
1,Pri-miR-96 RNA1 (Drosha processing site only),GGGAGAGGGUUUAUUUACGAAAGUAUAUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)Cc1cn(CCCNC(=...
2,Pri-miR-96 RNA2 (desired GG loop only),GGGAGAGGGUUUACGAACGAAAGUUGGUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)Cc1cn(CCCNC(=...
3,Pri-miR-96 RNA3 (both the Drosha target site a...,GGGAGAGGGUUUACGAUUUACGAAAGUAUAUGGUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)Cc1cn(CCCNC(=...
4,Pri-miR-96 RNA4 (GG to GC loop mutation),GGGAGAGGGUUUACGAUUUACGAAAGUAUAUCGUGGAUCCGCAAGG,Targaprimir-96,CCCN(CC(=O)N(CCC)CC(=O)N(CC(N)=O)Cc1cn(CCCNC(=...
...,...,...,...,...
141,Pre-miR-21,AGCCCCUGCCCACCGCACACUGCGCUGCCCCAGACCCACUGUGCGU...,SMB3,CCCN(CC(=O)N(CC(N)=O)Cc1cn(CCCCC(=O)CCCOc2ccc(...
142,Pre-miR-17/18a/20a,UAGCUUAUCAGACUGAUGUUGACUGUUGAAUCUCAUGGCAACACCA...,SMB4,CCCN(CC(=O)N(CC(N)=O)Cc1cn(CCCCC(=O)CCCOc2c(C(...
143,Pre-miR-210,AGCCCCUGCCCACCGCACACUGCGCUGCCCCAGACCCACUGUGCGU...,SMC1,CN1CCN(c2ccc3nc(-c4ccc5[nH]c(-c6cccc(OCCCC(=O)...
144,Pri-miR-96,GGGAGAGGGUUUAAUUACGAAAGUAAUUGGAUCCGCAAGG,SMC2,CCCN(CC(=O)N(CCC)C(=O)CCCOc1ccc(-c2nc3cc(N4CCN...


In [46]:
nuc_ok.to_csv("final_miRNA_interactions.csv", index=False)

In [47]:
repeats_annotation['check_rna_seq'] = repeats_annotation['rna_content'].map(set)
repeats_annotation['check_rna_seq'].value_counts() 

check_rna_seq
{G, U, C}       38
{G, C, U, A}    31
{G, C}          18
{C, U, A}        4
{A, G, C}        4
{T, G, C}        1
{U, A}           1
Name: count, dtype: int64

In [ ]:
repeats_annotation['check_smiles_seq'] = repeats_annotation['small_molecule_content'].map(set)
repeats_annotation['check_smiles_seq'].value_counts()

In [49]:
repeats_annotation['canonical_smiles'] = repeats_annotation['small_molecule_content'].map(try_canonical_smiles)
n_total = len(repeats_annotation)
n_valid = repeats_annotation['canonical_smiles'].notna().sum()
n_invalid = repeats_annotation['canonical_smiles'].isna().sum()

print(f"Total: {n_total}")
print(f"Valid: {n_valid}")

Total: 97
Valid: 97


In [50]:
repeats_annotation[repeats_annotation['canonical_smiles'].isna()]

,RNA_name,rna_content,small_molecule_name,small_molecule_content,kd,check_rna_seq,check_smiles_seq,canonical_smiles


In [52]:
repeats_annotation.drop(columns=['kd', 'check_rna_seq', 'check_smiles_seq', 'small_molecule_content'], inplace=True)
repeats_annotation.to_csv("final_repeats_interactions.csv", index=False)

In [54]:
ribosomal_annotation.rename(columns={
    "small_moleculecontent": "small_molecule_content"
}, inplace=True)

In [75]:
ribosomal_annotation['check_rna_seq'] = ribosomal_annotation['rna_content'].map(set)
ribosomal_annotation['check_rna_seq'].value_counts()
ribosomal_annotation['check_smiles_seq'] = ribosomal_annotation['small_molecule_content'].map(set)
ribosomal_annotation['check_smiles_seq'].value_counts()
ribosomal_annotation['canonical_smiles'] = ribosomal_annotation['small_molecule_content'].map(try_canonical_smiles)
n_total = len(ribosomal_annotation)
n_valid = ribosomal_annotation['canonical_smiles'].notna().sum()
n_invalid = ribosomal_annotation['canonical_smiles'].isna().sum()

In [76]:
print(f"Total: {n_total}")
print(f"Valid: {n_valid}")
print(f"Invalid: {n_invalid}")

Total: 295
Valid: 295
Invalid: 0


In [77]:
ribosomal_annotation[ribosomal_annotation['canonical_smiles'].isna()]

,RNA_name,rna_content,small_molecule_name,small_molecule_content,kd,check_rna_seq,check_smiles_seq,canonical_smiles


In [72]:
old = 'c1(c(cc2c(c1)nc(n2Cc1ccc(cc1N(=O)O)N(=O)O)C1CCNCC1)Cl)Cl'
new = 'O=[N+]([O-])c1ccc(Cn2c(C3CCNCC3)nc3cc(Cl)c(Cl)cc32)c([N+](=O)[O-])c1'

mask = ribosomal_annotation["small_molecule_content"].eq(old)

ribosomal_annotation.loc[mask, "small_molecule_content"] = new

In [73]:
old = 'c1(c(cc2c(c1)nc(n2Cc1ccc(cc1N(=O)O)C(F)(F)F)C1CCNCC1)Cl)Cl'
new = 'O=[N+]([O-])c1cc(C(F)(F)F)ccc1Cn1c(C2CCNCC2)nc2cc(Cl)c(Cl)cc21'

mask = ribosomal_annotation["small_molecule_content"].eq(old)

ribosomal_annotation.loc[mask, "small_molecule_content"] = new

In [74]:
old = 'c1c(cc2c(c1)[nH]c(n2)[C@H]1C[C@H](CCC1)N)N(=O)O'
new = 'N[C@H]1CCC[C@@H](c2nc3cc([N+](=O)[O-])ccc3[nH]2)C1'

mask = ribosomal_annotation["small_molecule_content"].eq(old)

ribosomal_annotation.loc[mask, "small_molecule_content"] = new

In [78]:
ribosomal_annotation.drop(columns=['kd', 'check_rna_seq', 'check_smiles_seq', 'small_molecule_content'], inplace=True)
ribosomal_annotation.to_csv("final_ribosomal_interactions.csv", index=False)

In [87]:
viral_annotation

,RNA_name,rna_content,small_molecule_name,small_molecule_content,kd,check_rna_seq,check_smiles_seq,canonical_smiles
0,RNA_Str_HIV-2 TAR,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,neocarzinostatin,CC1C(C(C(C(O1)OC2C3CC#CC4C(O4)(C#CC3=CC2OC(=O)...,4.853872,"{A, G, U, C}","{6, (, ), 1, 7, #, N, =, 3, 5, 4, 2, O, C}",CNC1C(OC2C(OC(=O)c3c(O)ccc4c(C)cc(OC)cc34)C=C3...
1,RNA_Str_HIV-2 TAR,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,Neocarzinostatin_chromophore_derivative_1,c12c(cc(cc1C)OC)[C@@]1(C(=O)C=C2)C(=O)O[C@@H]2...,4.853872,"{A, G, U, C}","{(, ), [, ], 1, H, N, c, =, @, 2, O, C}",CN[C@H]1[C@H](O[C@H]2c3cc4c(cc3[C@@H]3[C@H]2OC...
2,RNA_Str_HIV-2 TAR,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,Neocarzinostatin_chromophore_derivative_2,c12c(cccc1)[C@]1(C(=O)C=C2)[C@@H](O[C@@H]2[C@@...,4.853872,"{A, G, U, C}","{(, ), [, ], 1, H, N, c, =, @, 2, O, C}",N[C@@H]1[C@@H](O[C@@H]2O[C@H]3C(=O)c4cc5ccccc5...
3,RNA_Str_HIV-2 TAR,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,Neocarzinostatin_chromophore_derivative_3,c12c(cccc1)[C@]1([C@@H](C=C2)O[C@H]2[C@@H]([C@...,4.886057,"{A, G, U, C}","{(, ), [, ], 1, H, N, c, =, @, 2, O, C}",N[C@H]1[C@H](O[C@@H]2C=Cc3ccccc3[C@@]23C(=O)O[...
4,RNA_Str_HIV-2 TAR,GGCCAGAUUGAGCCUGGGAGCUCUCUGGCC,Neocarzinostatin_chromophore_derivative_5,c12c(cc(cc1)OC)[C@@]1(C(=O)C=C2)C=C[C@@H]2[C@@...,5.096910,"{A, G, U, C}","{(, ), [, ], 1, H, N, c, =, @, 2, O, C}",CN[C@@H]1[C@@H](O[C@@H]2c3cc4ccccc4cc3[C@H]3[C...
...,...,...,...,...,...,...,...,...
276,RRE4,GGUGGGCGCAGCUUCGGCUGCGGACACC,Paromomycin,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)O)N)OC3C(C(C(...,4.809668,"{A, G, U, C}","{(, ), 1, C, 3, 4, 2, O, N}",NCC1OC(OC2C(CO)OC(OC3C(O)C(N)CC(N)C3OC3OC(CO)C...
277,RRE11,GGUGGGCGCAGCUUCGGCUGCGCAACCACC,Paromomycin,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)O)N)OC3C(C(C(...,4.728158,"{A, G, U, C}","{(, ), 1, C, 3, 4, 2, O, N}",NCC1OC(OC2C(CO)OC(OC3C(O)C(N)CC(N)C3OC3OC(CO)C...
278,RRE3,GGUGGGCGCAGCUUCGGCUGCGCCCACC,Kanamycin B,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)N)O)O)OC3C(C(...,4.903090,"{A, G, U, C}","{(, ), 1, C, 3, 2, O, N}",NCC1OC(OC2C(N)CC(N)C(OC3OC(CO)C(O)C(N)C3O)C2O)...
279,RRE4,GGUGGGCGCAGCUUCGGCUGCGGACACC,Kanamycin B,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CO)O)N)O)O)OC3C(C(...,5.200659,"{A, G, U, C}","{(, ), 1, C, 3, 2, O, N}",NCC1OC(OC2C(N)CC(N)C(OC3OC(CO)C(O)C(N)C3O)C2O)...


In [93]:
viral_annotation['check_rna_seq'] = viral_annotation['rna_content'].map(set)
viral_annotation['check_rna_seq'].value_counts()
viral_annotation['check_smiles_seq'] = viral_annotation['small_molecule_content'].map(set)
viral_annotation['check_smiles_seq'].value_counts()
viral_annotation['canonical_smiles'] = viral_annotation['small_molecule_content'].map(try_canonical_smiles)
n_total = len(viral_annotation)
n_valid = viral_annotation['canonical_smiles'].notna().sum()
n_invalid = viral_annotation['canonical_smiles'].isna().sum()

In [94]:
print(f"Total: {n_total}")
print(f"Valid: {n_valid}")
print(f"Invalid: {n_invalid}")

Total: 281
Valid: 281
Invalid: 0


In [95]:
viral_annotation[viral_annotation['canonical_smiles'].isna()]

,RNA_name,rna_content,small_molecule_name,small_molecule_content,kd,check_rna_seq,check_smiles_seq,canonical_smiles


In [92]:
old = 'Coc1ccc2c(c1)c(Nc1ccc(c(c1)CN1CCN(CC1)C)O)c1c(n2)cc(cc1)Cl'
new = 'COc1ccc2nc3cc(Cl)ccc3c(Nc3ccc(O)c(CN4CCN(C)CC4)c3)c2c1'

mask = viral_annotation["small_molecule_content"].eq(old)

viral_annotation.loc[mask, "small_molecule_content"] = new

old = 'Ccc1nc2ccccc2cc1C(=O)O'
new = 'CCc1nc2ccccc2cc1C(=O)O'

mask = viral_annotation["small_molecule_content"].eq(old)

viral_annotation.loc[mask, "small_molecule_content"] = new

old = '[C@H]12[C@@H](C[C@@H]3[N@@](C1)(CCc1c3[nH]c3c1cccc3)C)[C@H]([C@H](CC2)O)C(=O)OC'
new = '[C@H]12[C@@H](C[C@@H]3[N@@+](C1)(CCc1c3[nH]c3c1cccc3)C)[C@H]([C@H](CC2)O)C(=O)OC'

mask = viral_annotation["small_molecule_content"].eq(old)

viral_annotation.loc[mask, "small_molecule_content"] = new

In [97]:
viral_annotation.drop(columns=['kd', 'check_rna_seq', 'check_smiles_seq', 'small_molecule_content'], inplace=True)
viral_annotation.to_csv("final_viral_interactions.csv", index=False)

In [99]:
riboswitch_annotation

,RNA_name,rna_content,small_molecule_name,small_molecule_content,kd,check_rna_seq,check_smiles_seq,canonical_smiles
0,ADENINE RIBOSWITCH,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,Adenine,C1=NC2=NC=NC(=C2N1)N,6.397940,"{G, C, U, A}","{(, ), 1, C, =, 2, N}",Nc1ncnc2nc[nH]c12
1,ADENINE RIBOSWITCH,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,2-aminopurine,C1=C2C(=NC(=N1)N)N=CN2,5.789147,"{G, C, U, A}","{(, ), 1, N, =, 2, C}",Nc1ncc2[nH]cnc2n1
2,ADENINE RIBOSWITCH,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,"2,4,6-triaminopyrimidine",C1=C(N=C(N=C1N)N)N,4.698970,"{G, C, U, A}","{(, ), 1, C, =, N}",Nc1cc(N)nc(N)n1
3,ADENINE RIBOSWITCH,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,"2,6- diaminopurine",C1=NC2=NC(=NC(=C2N1)N)N,7.698970,"{G, C, U, A}","{(, ), 1, C, =, 2, N}",Nc1nc(N)c2[nH]cnc2n1
4,ADENINE RIBOSWITCH,GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCAC...,"2,4-diaminopurine",C1=NC(=NC2(C1=NC=N2)N)N,8.397940,"{G, C, U, A}","{(, ), 1, C, =, 2, N}",NC1=NC2(N)N=CN=C2C=N1
...,...,...,...,...,...,...,...,...
95,Tetracycline riboswitch A50U mutant,GGGCCUAAAACAUACCAGAUCGCCACCCGCGCUUUAAUCUGGAGAG...,dox,C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC...,4.978811,"{A, G, U, C}","{], (, [, ), 1, H, N, =, @, 3, 5, 4, 2, O, C}",COc1cccc2c1C(=O)c1c(O)c3c(c(O)c1C2=O)C[C@@](O)...
96,FMN RIBOSWITCH APTAMER,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,Ribocil,CNC1=NC=C(C=N1)CN2CCC[C@@H](C2)C3=NC(=CC(=O)N3...,7.795880,"{A, G, U, C}","{(, ), [, ], 1, C, H, S, =, @, 3, 4, 2, O, N}",CNc1ncc(CN2CCC[C@H](c3nc(-c4cccs4)cc(=O)[nH]3)...
97,FMN RIBOSWITCH APTAMER,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,Ribocil-A,CNc1ncc(cn1)CN1CCC[C@H](C1)c1nc(O)cc(n1)c1cccs1,5.000000,"{A, G, U, C}","{s, (, ), [, ], 1, C, H, c, @, n, O, N}",CNc1ncc(CN2CCC[C@@H](c3nc(O)cc(-c4cccs4)n3)C2)cn1
98,FMN RIBOSWITCH APTAMER,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,Ribocil-B,CNC1=NC=C(C=N1)CN2CCC[C@@H](C2)C3=NC(=CC(=O)N3...,8.180456,"{A, G, U, C}","{(, ), [, ], 1, C, H, S, =, @, 3, 4, 2, O, N}",CNc1ncc(CN2CCC[C@H](c3nc(-c4cccs4)cc(=O)[nH]3)...


In [108]:
riboswitch_annotation['check_rna_seq'] = riboswitch_annotation['rna_content'].map(set)
riboswitch_annotation['check_rna_seq'].value_counts()
riboswitch_annotation['check_smiles_seq'] = riboswitch_annotation['small_molecule_content'].map(set)
riboswitch_annotation['check_smiles_seq'].value_counts()
riboswitch_annotation['canonical_smiles'] = riboswitch_annotation['small_molecule_content'].map(try_canonical_smiles)
n_total = len(riboswitch_annotation)
n_valid = riboswitch_annotation['canonical_smiles'].notna().sum()
n_invalid = riboswitch_annotation['canonical_smiles'].isna().sum()

In [109]:
print(f"Total: {n_total}")
print(f"Valid: {n_valid}")
print(f"Invalid: {n_invalid}")

Total: 100
Valid: 100
Invalid: 0


In [110]:
riboswitch_annotation[riboswitch_annotation['canonical_smiles'].isna()]

,RNA_name,rna_content,small_molecule_name,small_molecule_content,kd,check_rna_seq,check_smiles_seq,canonical_smiles


In [107]:
old = 'Ccn1c2ccc(cc2c2c1cccc2)N1CCNCC1'
new = 'CC[n+]1ccc2c(c1)ccc3c(N4CCNCC4)cccc23'

mask = riboswitch_annotation["small_molecule_content"].eq(old)

riboswitch_annotation.loc[mask, "small_molecule_content"] = new

old = 'n1c(ncc(c1N)CN1=CC=CC(=C1)CCO[P@@](=O)(O)P(=O)(O)O)C'
new = 'n1c(ncc(c1N)C[n+]1=CC=CC(=C1)CCOP(=O)(O)P(=O)(O)O)C'

mask = riboswitch_annotation["small_molecule_content"].eq(old)

riboswitch_annotation.loc[mask, "small_molecule_content"] = new


In [111]:
riboswitch_annotation.drop(columns=['kd', 'check_rna_seq', 'check_smiles_seq', 'small_molecule_content'], inplace=True)
riboswitch_annotation.to_csv("final_riboswitch_interactions.csv", index=False)